In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Hyperparameters & Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
epochs = 5
batch_size = 64
learning_rate = 0.001

# 2. Data Loading and Augmentation
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                            shuffle=True, num_workers=2)

# 3. Model Architecture
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) # Output: 16x16
        x = self.pool(F.relu(self.conv2(x))) # Output: 8x8
        x = x.view(-1, 64 * 8 * 8)           # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN().to(device)

# 4. Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 5. Training Loop
print(f"Starting training on {device}...")
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 200 == 199:    # Print every 200 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 200:.3f}')
            running_loss = 0.0

print("Finished Training")

# 6. Save the Model Weights
PATH = './app/model.pth'
torch.save(model.state_dict(), PATH)
print(f"Model weights saved to {PATH}")

Starting training on cuda...
[1,   200] loss: 1.626
[1,   400] loss: 1.273
[1,   600] loss: 1.144
[2,   200] loss: 0.941
[2,   400] loss: 0.913
[2,   600] loss: 0.869
[3,   200] loss: 0.708
[3,   400] loss: 0.711
[3,   600] loss: 0.697
[4,   200] loss: 0.523
[4,   400] loss: 0.539
[4,   600] loss: 0.527
[5,   200] loss: 0.343
[5,   400] loss: 0.368
[5,   600] loss: 0.395
Finished Training
Model weights saved to ./app/model.pth
